In [29]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [30]:
# Конструиране на GraphSAGE модела с Mean Aggregation
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGEMean(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels,
            aggr="mean"
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            out_channels,
            aggr="mean"
        )

        self.dropout = 0.5

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [31]:
# Конструиране на GraphSAGE модела с Max Aggregation
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGEMax(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels,
            aggr="max"
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            out_channels,
            aggr="max"
        )

        self.dropout = 0.5

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [32]:
# Създаване на модела с Mean Aggregation
model = GraphSAGEMean(
    dataset.num_features,
    64,
    dataset.num_classes)

In [33]:
# Създаване на модела с Max Aggregation
model = GraphSAGEMax(
    dataset.num_features,
    64,
    dataset.num_classes
)

In [34]:
# Конструиране на GIN модела
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.nn import GINConv

class GIN(torch.nn.Module):

    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        mlp1 = Sequential(
            Linear(in_channels, hidden_channels),
            ReLU(),
            Linear(hidden_channels, hidden_channels)
        )

        mlp2 = Sequential(
            Linear(hidden_channels, hidden_channels),
            ReLU(),
            Linear(hidden_channels, out_channels)
        )

        self.conv1 = GINConv(mlp1)

        self.conv2 = GINConv(mlp2)

        self.dropout = 0.5

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [38]:
# Създаване на GIN модела
model = GIN(
    dataset.num_features,
    64,
    dataset.num_classes
)

In [39]:
# Инициализиране на оптимизатора
import torch

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [40]:
# Функция за обучение
import torch.nn.functional as F

def train():

    model.train()

    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [41]:
# Обучение на модела и измерване на времето
import time

start_time = time.time()

for epoch in range(1, 201):

    loss = train()

    if epoch % 20 == 0:
        print(
            f"Epoch: {epoch:3d}, "
            f"Loss: {loss:.4f}"
        )

training_time = time.time() - start_time

print(f"\nВреме за обучение: {training_time:.2f} s")

Epoch:  20, Loss: 0.0729
Epoch:  40, Loss: 0.0386
Epoch:  60, Loss: 0.0129
Epoch:  80, Loss: 0.0855
Epoch: 100, Loss: 0.0003
Epoch: 120, Loss: 0.0005
Epoch: 140, Loss: 0.0013
Epoch: 160, Loss: 0.0063
Epoch: 180, Loss: 0.0000
Epoch: 200, Loss: 0.0001

Време за обучение: 10.91 s


In [42]:
# Оценяване
from sklearn.metrics import accuracy_score, f1_score
def evaluate():

    model.eval()

    with torch.no_grad():

        out = model(data.x, data.edge_index)

        pred = out.argmax(dim=1)

    y_true = data.y[data.test_mask].cpu()
    y_pred = pred[data.test_mask].cpu()

    accuracy = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    return accuracy, f1

In [43]:
# Изчисляване на броя параметри
num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Брой параметри: {num_params}")

Брой параметри: 100551


In [44]:
# Извеждане на резултатите
accuracy, f1 = evaluate()

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1-score: {f1:.4f}")
print(f"Параметри     : {num_params}")
print(f"Време         : {training_time:.2f} s")

Accuracy      : 0.6910
Macro F1-score: 0.6958
Параметри     : 100551
Време         : 10.91 s


In [45]:
model

GIN(
  (conv1): GINConv(nn=Sequential(
    (0): Linear(in_features=1433, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  ))
  (conv2): GINConv(nn=Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=7, bias=True)
  ))
)